<a href="https://colab.research.google.com/github/jdmartinev/CVBootcampMCDA/blob/main/notebooks/02_image_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Workshop CLIP — Notebook 02: Image Retrieval

---

En este notebook construyes el motor de búsqueda completo. Al terminarlo tendrás:

- Un índice vectorial de 800 imágenes de Flickr30k
- Búsqueda **texto → imagen** con exploración interactiva
- Búsqueda **imagen → imagen** (cross-modal)
- Análisis de casos de fallo de CLIP

**TODOs en este notebook:** 3  
**Prerequisito:** haber completado `01_clip_baseline.ipynb` — copia tus 4 funciones en la celda de Setup.

> ⚠️ La indexación de 800 imágenes tarda ~2 min en CPU. Ejecútala una sola vez.

---
## 0 — Setup

In [ ]:
%%capture
!pip install transformers datasets Pillow ipywidgets pandas

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import random
from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor
from datasets import load_dataset
import ipywidgets as widgets
from IPython.display import display, clear_output

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {device}")

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"
clip_model = CLIPModel.from_pretrained(MODEL_ID).to(device)
clip_processor = CLIPProcessor.from_pretrained(MODEL_ID)
clip_model.eval()
print("✅ Modelo cargado")

### 0.3 — Pega aquí tus funciones del notebook 01

Copia las cuatro funciones que implementaste: `get_text_embeddings`, `get_image_embeddings`, `compute_similarity_matrix`, `compute_scores`.

In [ ]:
# ── Pega aquí tus funciones del notebook 01 ────────────────────────────────────

def get_text_embeddings(texts, model, processor, device):
    raise NotImplementedError("Copia tu implementación del NB01")

def get_image_embeddings(images, model, processor, device):
    raise NotImplementedError("Copia tu implementación del NB01")

def compute_similarity_matrix(image_embs, text_embs, tau=1.0):
    raise NotImplementedError("Copia tu implementación del NB01")

def compute_scores(query_emb, corpus_emb):
    raise NotImplementedError("Copia tu implementación del NB01")


# ── Verificación rápida ────────────────────────────────────────────────────────
# q = F.normalize(torch.randn(1, 512), dim=-1)
# C = F.normalize(torch.randn(5, 512), dim=-1)
# assert compute_scores(q, C).shape == (5,)
# print("✅ Funciones del NB01 OK")

---
## 1 — Cargar el corpus completo

Cargamos 800 imágenes del split `test` de Flickr30k con la misma semilla que en el notebook 01 para reproducibilidad.

In [ ]:
CORPUS_SIZE = 800
SEED = 42

print("Cargando Flickr30k...")
raw = load_dataset("AnyModal/flickr30k", split="test")

random.seed(SEED)
all_indices = list(range(len(raw)))
random.shuffle(all_indices)
corpus_indices = all_indices[:CORPUS_SIZE]

corpus_images   = [raw[i]["image"].convert("RGB") for i in tqdm(corpus_indices, desc="Cargando")]
corpus_captions = [raw[i]["original_alt_text"] for i in corpus_indices]
corpus_ids      = [str(raw[i]["img_id"]) for i in corpus_indices]

print(f"\n✅ Corpus: {len(corpus_images)} imágenes | {len(corpus_ids)} IDs únicos")

---
## 2 — Construir el índice de imágenes

El índice es simplemente la matriz de embeddings de todas las imágenes del corpus. Una vez construido, **no necesitamos las imágenes originales para buscar** — todo el retrieval ocurre en el espacio de embeddings.

```
corpus_images (800 PIL) ──► build_image_index ──► image_index (800, 512)
```

Procesamos en batches de 32 para eficiencia: en lugar de hacer 800 llamadas al modelo, hacemos 25.

### ✏️ TODO 5 — `build_image_index`

Construye la matriz de embeddings procesando el corpus en batches.

**Pasos:**
1. Itera en batches: `for i in tqdm(range(0, len(images), batch_size), desc="Indexando")`
2. Extrae el batch: `batch = images[i : i + batch_size]`
3. Obtén embeddings con `get_image_embeddings(batch, model, processor, device)`
4. Acumula en una lista
5. Concatena con `torch.cat(lista, dim=0)` y retorna

> 💡 El resultado debe ser un tensor `(N, 512)` normalizado L2 — lo verificamos porque `get_image_embeddings` ya normaliza.

In [ ]:
def build_image_index(
    images: list,
    model: CLIPModel,
    processor: CLIPProcessor,
    device: torch.device,
    batch_size: int = 32,
) -> torch.Tensor:
    """
    Indexa el corpus extrayendo embeddings CLIP en batches.

    Args:
        images:     lista de N PIL Images
        batch_size: imágenes por llamada al modelo

    Returns:
        Tensor (N, 512) normalizado L2.
    """
    # TODO 5
    raise NotImplementedError


# ── Construir el índice ────────────────────────────────────────────────────────
# image_index = build_image_index(corpus_images, clip_model, clip_processor, device)
# assert image_index.shape == (CORPUS_SIZE, 512)
# norms = image_index.norm(dim=-1)
# assert torch.allclose(norms, torch.ones(CORPUS_SIZE), atol=1e-4)
# print(f"✅ Índice construido: {image_index.shape}")

---
## 3 — Espacio compartido: distribución de similitudes

Una forma directa de verificar que el espacio compartido de CLIP funciona es comparar la similitud coseno entre:

- **Pares correctos:** imagen $i$ con su propio caption → deberían tener similitud **alta**
- **Pares aleatorios:** imagen $i$ con un caption de otra imagen → deberían tener similitud **baja**

Si las dos distribuciones están bien separadas, significa que CLIP aprendió a alinear modalidades correctamente y que `compute_scores` puede distinguir la imagen correcta del resto.

In [ ]:
VIZ_N = 300

first_captions = [caps[0] for caps in corpus_captions[:VIZ_N]]
txt_embs = get_text_embeddings(first_captions, clip_model, clip_processor, device)
img_embs = image_index[:VIZ_N]

S = (img_embs @ txt_embs.T).numpy()

correct_sims = np.diag(S)
rng = np.random.default_rng(SEED)
mask = ~np.eye(VIZ_N, dtype=bool)
random_sims = rng.choice(S[mask], size=2000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.hist(random_sims, bins=40, alpha=0.6, color="steelblue", label="Pares aleatorios", density=True)
ax.hist(correct_sims, bins=30, alpha=0.7, color="tomato", label="Pares correctos", density=True)
ax.axvline(correct_sims.mean(), color="tomato", linestyle="--", linewidth=1.5,
           label=f"Media correctos: {correct_sims.mean():.3f}")
ax.axvline(random_sims.mean(), color="steelblue", linestyle="--", linewidth=1.5,
           label=f"Media aleatorios: {random_sims.mean():.3f}")
ax.set_xlabel("Similitud coseno", fontsize=11)
ax.set_ylabel("Densidad", fontsize=11)
ax.set_title("Distribución de similitudes", fontsize=12)
ax.legend(fontsize=9)

ax2 = axes[1]
im = ax2.imshow(S[:30, :30], cmap="RdYlGn", vmin=0.0, vmax=0.5)
plt.colorbar(im, ax=ax2, label="Similitud coseno")
ax2.set_xlabel("Texto (caption)", fontsize=11)
ax2.set_ylabel("Imagen", fontsize=11)
ax2.set_title("Matriz de similitud (30×30)\nDiagonal = pares correctos", fontsize=12)
for i in range(30):
    ax2.add_patch(plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                                 fill=False, edgecolor="black", linewidth=1.2))

plt.suptitle("¿Funciona el espacio compartido de CLIP?", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

gap = correct_sims.mean() - random_sims.mean()
print(f"Gap (correctos - aleatorios): {gap:.4f}")
print(f"→ Un gap positivo confirma que CLIP separa pares correctos de incorrectos.")
print(f"  Este gap es lo que hace posible el retrieval.")

---
## 4 — Búsqueda texto → imagen

Con el índice construido, el pipeline de búsqueda es:

```
texto ──► get_text_embeddings ──► compute_scores(query, index) ──► top-k ──► imágenes
```

Todo el retrieval sobre 800 imágenes toma ~10ms — el costo está en la indexación, no en la búsqueda.

### ✏️ TODO 6 — `search_by_text`

Implementa el pipeline completo de búsqueda por texto.

**Pasos:**
1. Codifica el texto con `get_text_embeddings([query_text], model, processor, device)`
2. Calcula scores con `compute_scores(text_emb, image_index)`
3. Obtén top-k índices: `torch.topk(scores, k=top_k).indices`
4. Extrae los scores correspondientes
5. Retorna `(indices, scores_top_k)`

In [ ]:
def search_by_text(
    query_text: str,
    image_index: torch.Tensor,
    model: CLIPModel,
    processor: CLIPProcessor,
    device: torch.device,
    top_k: int = 6,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Recupera las top_k imágenes más similares a un texto.

    Returns:
        (indices, scores) — Tensors de shape (top_k,)
    """
    # TODO 6
    raise NotImplementedError


# ── Función de visualización (dada) ───────────────────────────────────────────
def show_results(
    query_label: str,
    indices: torch.Tensor,
    scores: torch.Tensor,
    images: list,
    ids: list,
    captions: list = None,
):
    k = len(indices)
    fig, axes = plt.subplots(1, k, figsize=(3 * k, 3.8))
    if k == 1: axes = [axes]
    fig.suptitle(f'Query: "{query_label}"', fontsize=12, fontweight="bold", y=1.02)
    for rank, (idx, score) in enumerate(zip(indices.tolist(), scores.tolist())):
        ax = axes[rank]
        ax.imshow(images[idx])
        title = f"#{rank+1}  sim={score:.3f}"
        if captions:
            title += f"\n{captions[idx][0][:35]}..."
        ax.set_title(title, fontsize=7.5)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


# ── Prueba (descomenta cuando TODO 6 esté listo) ───────────────────────────────
# idx, scores = search_by_text(
#     "two dogs running on the beach",
#     image_index, clip_model, clip_processor, device
# )
# show_results("two dogs running on the beach", idx, scores, corpus_images, corpus_ids, corpus_captions)

### 4.2 — Exploración interactiva

Prueba distintos tipos de query:
- Objetos: `"a red bicycle"`
- Escenas: `"a snowy mountain landscape"`
- Acciones: `"children playing soccer"`
- Emociones/atmósfera: `"a joyful crowd"`

In [ ]:
text_box = widgets.Text(
    value="a child playing in the park",
    placeholder="Escribe tu query...",
    description="Query:",
    layout=widgets.Layout(width="480px"),
)
k_slider = widgets.IntSlider(value=6, min=1, max=10, step=1, description="top-k:",
                              layout=widgets.Layout(width="280px"))
btn = widgets.Button(description="Buscar 🔍", button_style="primary")
out = widgets.Output()

def on_search(b):
    with out:
        clear_output(wait=True)
        idx, sc = search_by_text(
            text_box.value, image_index,
            clip_model, clip_processor, device, top_k=k_slider.value
        )
        show_results(text_box.value, idx, sc, corpus_images, corpus_ids, corpus_captions)

btn.on_click(on_search)
display(widgets.VBox([widgets.HBox([text_box, k_slider, btn]), out]))

---
## 5 — Búsqueda imagen → imagen

Como texto e imágenes comparten el mismo espacio de embeddings, podemos usar una imagen como query directamente. El pipeline es idéntico — solo cambia la función de codificación del query.

Esto es lo que hace **Google Lens**: extraer el embedding de la imagen de entrada y recuperar las más similares del índice.

### ✏️ TODO 7 — `search_by_image`

Pipeline de búsqueda usando una imagen PIL como query. La estructura es idéntica a `search_by_text`, solo cambia el encoder del query.

> 💡 Una imagen puede aparecer en el propio corpus — si el índice fue construido con `corpus_images`, la imagen query podría tener similitud 1.0 consigo misma. Excluye ese caso si el argumento `exclude_idx` está presente: pon `scores[exclude_idx] = -1.0` antes de hacer `topk`.

In [ ]:
def search_by_image(
    query_image: Image.Image,
    image_index: torch.Tensor,
    model: CLIPModel,
    processor: CLIPProcessor,
    device: torch.device,
    top_k: int = 6,
    exclude_idx: int = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Recupera las top_k imágenes más similares a una imagen query.

    Args:
        query_image: PIL Image
        exclude_idx: índice a excluir del resultado (útil si la query está en el corpus)

    Returns:
        (indices, scores) — Tensors de shape (top_k,)
    """
    # TODO 7
    raise NotImplementedError


# ── Prueba (descomenta cuando TODO 7 esté listo) ───────────────────────────────
# query_img = corpus_images[0]
# idx, scores = search_by_image(
#     query_img, image_index, clip_model, clip_processor, device,
#     top_k=5, exclude_idx=0
# )
# fig, axes = plt.subplots(1, 6, figsize=(18, 3.5))
# axes[0].imshow(query_img); axes[0].set_title("QUERY", fontweight="bold", color="crimson"); axes[0].axis("off")
# for r, (i, s) in enumerate(zip(idx.tolist(), scores.tolist())):
#     axes[r+1].imshow(corpus_images[i])
#     axes[r+1].set_title(f"#{r+1}\n{s:.3f}", fontsize=8)
#     axes[r+1].axis("off")
# plt.suptitle("Búsqueda imagen → imagen", fontsize=13)
# plt.tight_layout(); plt.show()

### 5.2 — Exploración interactiva: imagen como query

In [ ]:
slider  = widgets.IntSlider(value=0, min=0, max=CORPUS_SIZE-1, step=1,
                             description="Imagen:",
                             layout=widgets.Layout(width="480px"))
img_btn = widgets.Button(description="Buscar similares 🖼️", button_style="primary")
preview = widgets.Output()
img_out = widgets.Output()

def update_preview(change):
    with preview:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(2.5, 2.5))
        ax.imshow(corpus_images[change["new"]])
        ax.set_title(f"Query [{change['new']}]", fontsize=9)
        ax.axis("off")
        plt.tight_layout(); plt.show()

def on_img_search(b):
    qi = slider.value
    with img_out:
        clear_output(wait=True)
        idx, sc = search_by_image(
            corpus_images[qi], image_index,
            clip_model, clip_processor, device,
            top_k=6, exclude_idx=qi
        )
        show_results(f"imagen[{qi}]", idx, sc, corpus_images, corpus_ids, corpus_captions)

slider.observe(update_preview, names="value")
img_btn.on_click(on_img_search)
update_preview({"new": 0})
display(widgets.VBox([widgets.HBox([slider, img_btn]), preview, img_out]))

---
## 6 — Casos de fallo

CLIP tiene limitaciones sistemáticas. Ejecuta las queries a continuación y observa los resultados.

In [ ]:
failure_queries = [
    "a street with no cars",           # negación
    "exactly three people",            # conteo exacto
    "a dog to the left of a cat",      # relación espacial
    "a red ball next to a blue box",   # atributos compuestos
]

for q in failure_queries:
    idx, sc = search_by_text(q, image_index, clip_model, clip_processor, device, top_k=4)
    show_results(q, idx, sc, corpus_images, corpus_ids)

### 6.2 — Reflexión (completa esta celda Markdown)

Para cada tipo de fallo, explica brevemente por qué crees que ocurre y en qué aplicación real sería crítico.

| Tipo de fallo | ¿Qué observaste? | ¿Por qué ocurre? | Aplicación crítica |
|---------------|-----------------|-----------------|-------------------|
| Negaciones | | | |
| Conteo exacto | | | |
| Relaciones espaciales | | | |
| Atributos compuestos | | | |

---
## ✅ Checkpoint

Verifica antes de pasar al notebook 03:

| Función | TODO | ¿Lista? |
|---------|------|---------|
| `build_image_index` | 5 | ☐ |
| `search_by_text` | 6 | ☐ |
| `search_by_image` | 7 | ☐ |
| `image_index` construido y verificado | — | ☐ |

En `03_competition_submission.ipynb` usarás `search_by_text`, `search_by_image`, `image_index` y `corpus_ids`. **Copia todo lo necesario en la celda de setup del notebook 03.**